# 01 — Análise Exploratória (EDA) do Churn

Objetivo: entender o dataset sintético de clientes, validar que o **sinal de negócio** existe (contrato, tenure, cobrança, suporte) e levantar hipóteses para a modelagem.

> Os dados são gerados por `src/data/generate_synthetic.py` a partir de um modelo logístico latente — ou seja, o churn tem causas interpretáveis, não é ruído.

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from src.data.generate_synthetic import generate_customers

df = generate_customers(n=8000, seed=42)
df.head()

## Visão geral e taxa de churn

In [ ]:
print(df.shape)
print(df.dtypes)
print(f"Taxa de churn global: {df['churn'].mean():.1%}")
df.describe()

## Churn por tipo de contrato

Hipótese de negócio: contratos mês a mês têm barreira de saída baixa e devem churnar muito mais.

In [ ]:
by_contract = df.groupby('contract')['churn'].mean().sort_values(ascending=False)
by_contract.plot(kind='bar', title='Taxa de churn por contrato', ylabel='taxa de churn')
plt.tight_layout(); plt.show()
by_contract

## Churn por tempo de casa (tenure)

Clientes novos ainda não criaram vínculo — esperamos churn decrescente com o tenure.

In [ ]:
df['faixa_tenure'] = pd.cut(df['tenure_months'], bins=[0,12,24,48,72],
                            labels=['0-12','13-24','25-48','49-72'], include_lowest=True)
by_tenure = df.groupby('faixa_tenure', observed=True)['churn'].mean()
by_tenure.plot(kind='bar', title='Churn por faixa de tenure', ylabel='taxa de churn')
plt.tight_layout(); plt.show()
by_tenure

## Cobrança mensal e serviço de internet

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.boxplot(column='monthly_charges', by='churn', ax=ax[0])
ax[0].set_title('Cobrança mensal x churn')
df.groupby('internet_service')['churn'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=ax[1], title='Churn por internet')
plt.suptitle(''); plt.tight_layout(); plt.show()

## Correlação das variáveis numéricas com o churn

In [ ]:
num = ['tenure_months','monthly_charges','total_charges','senior_citizen',
       'partner','dependents','tech_support','online_security','churn']
df[num].corr()['churn'].sort_values()

## Insights de negócio

1. **Contrato mês a mês é o maior fator de churn** — clientes sem fidelização saem muito mais. Ação: incentivar migração para contratos anuais.
2. **Churn cai fortemente com o tenure** — o risco se concentra nos primeiros 12 meses. Ação: onboarding e acompanhamento reforçados no início.
3. **Cobrança mensal alta e fibra ótica** elevam o churn — percepção de custo-benefício. Ação: revisar pricing/valor entregue na fibra.
4. **Suporte técnico e segurança online retêm** — quem tem esses serviços churna menos. Ação: oferecer esses add-ons como retenção.
5. **Pagamento por cheque eletrônico** correlaciona com churn (proxy de perfil menos engajado).

Esses achados guiam a feature engineering (`charges_per_tenure`, buckets de tenure) e explicam a importância que o SHAP atribui às variáveis na Etapa 5.